# 23. Du notebook a l'application - servir le RAG de la serie

**Navigation** : [<< Précédent](22c_DSPy_Prompt_Compile.ipynb) | [Index](README.md) | [Suivant >>](README.md)

Les carnets precedents construisent des pipelines qui vivent dans le notebook : les fonctions, l'etat et les resultats partagent les memes cellules. Cette serie s'arrete la ou commence l'usage reel : **servir** un pipeline a d'autres utilisateurs qu'un lecteur de notebook.

Ce carnet prend le pipeline RAG du carnet 05 (corpus Lincoln-Douglas, chunking, embeddings, recherche k-NN, generation avec citations) et lui fait franchir les trois etapes du passage au service :

1. **Extraire** - le pipeline devient un ensemble de fonctions regroupees et nommees ; le client et le modele d'embedding restent des globales **visibles** du carnet, heritees du 05 ;
2. **Servir** - une API locale FastAPI expose la requete RAG, testee par des appels HTTP reels ;
3. **Interfacer** - une UI Gradio parle a cette API - jamais au pipeline directement.

Le fil rouge mesure ce que chaque etape coute : le transport pur HTTP, la fonction directe, l'appel API, sur le meme lot de questions.

## Le pipeline hote

La serie ne possede pas de module RAGE partage (`GenAI/shared/helpers/` porte logging, configuration et sorties d'images - verifie). L'organe du pipeline est donc le carnet 05 lui-meme : l'extraction ci-dessous copie **fidelement** ses cellules clés, chacune identifiee par sa cellule d'origine. Un point d'arret honnete : si le 05 evolue, cette extraction doit le suivre - le cout est le prix de l'absence de module, et il se paie ici de maniere visible.

In [1]:
# Extrait fidele de 05_RAG_Modern.ipynb (cellule 0) - parametre de serie
# Parameters
BATCH_MODE = "true"


In [2]:
# Extrait fidele de 05_RAG_Modern.ipynb (cellule 5) - configuration du client (pattern dotenv de la serie)
# Import guards - verification des dependances
try:
    import openai
    OPENAI_AVAILABLE = True
except ImportError:
    OPENAI_AVAILABLE = False

try:
    import anthropic
    ANTHROPIC_AVAILABLE = True
except ImportError:
    ANTHROPIC_AVAILABLE = False

try:
    import requests
    REQUESTS_AVAILABLE = True
except ImportError:
    REQUESTS_AVAILABLE = False

try:
    import matplotlib
    MATPLOTLIB_AVAILABLE = True
except ImportError:
    MATPLOTLIB_AVAILABLE = False

try:
    import numpy
    NUMPY_AVAILABLE = True
except ImportError:
    NUMPY_AVAILABLE = False

try:
    import pandas
    PANDAS_AVAILABLE = True
except ImportError:
    PANDAS_AVAILABLE = False

try:
    from PIL import Image
    PIL_AVAILABLE = True
except ImportError:
    PIL_AVAILABLE = False

try:
    import IPython
    IPYTHON_AVAILABLE = True
except ImportError:
    IPYTHON_AVAILABLE = False

try:
    from dotenv import load_dotenv
    DOTENV_AVAILABLE = True
except ImportError:
    DOTENV_AVAILABLE = False

try:
    import torch
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False

try:
    import transformers
    TRANSFORMERS_AVAILABLE = True
except ImportError:
    TRANSFORMERS_AVAILABLE = False

try:
    import cv2
    CV2_AVAILABLE = True
except ImportError:
    CV2_AVAILABLE = False

try:
    import pydantic
    PYDANTIC_AVAILABLE = True
except ImportError:
    PYDANTIC_AVAILABLE = False

# Resume des dependances disponibles
_DEPS = {
    'openai': OPENAI_AVAILABLE,
    'anthropic': ANTHROPIC_AVAILABLE,
    'requests': REQUESTS_AVAILABLE,
    'matplotlib': MATPLOTLIB_AVAILABLE,
    'numpy': NUMPY_AVAILABLE,
    'pandas': PANDAS_AVAILABLE,
    'PIL': PIL_AVAILABLE,
    'IPython': IPYTHON_AVAILABLE,
    'dotenv': DOTENV_AVAILABLE,
    'torch': TORCH_AVAILABLE,
    'transformers': TRANSFORMERS_AVAILABLE,
    'cv2': CV2_AVAILABLE,
    'pydantic': PYDANTIC_AVAILABLE,
}
available = [k for k, v in _DEPS.items() if v]
missing = [k for k, v in _DEPS.items() if not v]
print(f"Dependances: {len(available)}/{len(_DEPS)} disponibles"
      + (f" | Manquantes: {missing}" if missing else ""))

from pathlib import Path
import os
import json
import numpy as np
import pandas as pd
from typing import List, Dict, Any
from dotenv import load_dotenv
from openai import OpenAI

# Charger les variables d'environnement
# Chargement robuste de la configuration .env
from dotenv import load_dotenv
import os
# Recherche du .env dans tous les parents (pour Papermill qui change le cwd)
current_path = Path.cwd()
env_loaded = False
for _ in range(10):
    env_path = current_path / ".env"
    if env_path.exists():
        load_dotenv(env_path)
        print(f".env charge depuis: {env_path.name}")
        env_loaded = True
        break
    if current_path.name == "GenAI" or len(current_path.parts) <= 1:
        break
    current_path = current_path.parent
if not env_loaded:
    print("WARNING: .env non trouve, utilisation variables environnement")

# Mode batch pour tests automatisés
BATCH_MODE = os.getenv("BATCH_MODE", "false").lower() == "true"

# Client OpenAI
client = OpenAI()

# Modèle par défaut depuis .env
DEFAULT_MODEL = os.getenv("OPENAI_MODEL", "gpt-5-mini")

print(f"Configuration chargée - Mode batch: {BATCH_MODE}")
print(f"Modèle embeddings: text-embedding-3-large")
print(f"Modèle génération: {DEFAULT_MODEL}")

Dependances: 11/13 disponibles | Manquantes: ['transformers', 'cv2']
.env charge depuis: .env


Configuration chargée - Mode batch: False
Modèle embeddings: text-embedding-3-large
Modèle génération: gpt-5-mini


In [3]:
# Dependances du passage au service - pre-provisionnees
import threading, socket, time, statistics
import httpx
import uvicorn
from fastapi import FastAPI
from fastapi.testclient import TestClient
import gradio as gr
print("gradio", gr.__version__, "| fastapi", __import__("fastapi").__version__)

gradio 6.30.0 | fastapi 0.141.1


### Lecture de la configuration

Meme contrat que le 05 : la cle vient de l'environnement (`.env`), jamais du code. Les dependances de service (FastAPI, Uvicorn, Gradio) sont locales au carnet : rien ne deploie, tout s'arrete en fin d'execution.

In [4]:
# Extrait fidele de 05_RAG_Modern.ipynb (cellule 12) - corpus : fetch du debat Lincoln-Douglas
import requests
from bs4 import BeautifulSoup

def fetch_debate_text() -> str:
    """Récupère le texte du premier débat Lincoln-Douglas."""
    url = "https://home.nps.gov/liho/learn/historyculture/debate1.htm"
    
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, "html.parser")
        main_div = soup.select_one("div.ColumnMain")
        
        if main_div:
            return main_div.get_text(separator="\n", strip=True)
        else:
            raise ValueError("Conteneur principal non trouvé")
            
    except Exception as e:
        print(f"Erreur lors du scraping: {e}")
        # Texte de fallback pour le mode batch
        return """Lincoln-Douglas Debate - Ottawa, Illinois, August 21, 1858
        
Abraham Lincoln argued that slavery was morally wrong and should not be extended 
into new territories. He believed in the principle that "all men are created equal" 
as stated in the Declaration of Independence.

Lincoln stated: "I have no purpose to introduce political and social equality 
between the white and black races. There is a physical difference between the two, 
which, in my judgment, will probably forever forbid their living together upon the 
footing of perfect equality."

However, Lincoln maintained that Black Americans had the right to earn their own 
bread and improve their condition. He opposed the expansion of slavery while 
acknowledging the constitutional protections it had in existing states.

Douglas advocated for popular sovereignty, allowing each territory to decide 
the slavery question for itself. He accused Lincoln of wanting to make all 
states free states, which Lincoln denied.

The debate centered on the interpretation of the Dred Scott decision and 
whether Congress could prohibit slavery in the territories."""

# Récupération du texte
debate_text = fetch_debate_text()
print(f"Texte récupéré: {len(debate_text)} caractères")
print(f"Aperçu: {debate_text[:500]}...")

Texte récupéré: 98529 caractères
Aperçu: First Debate: Ottawa, Illinois
August 21, 1858
It was dry and dusty, between 10,000 and 12,000 people were in attendance when the debate began at 2:00 p.m. There were no seats or bleachers.
Douglas charged Lincoln with trying to “abolitionize” the Whig and Democratic Parties. He also charged Lincoln had been present when a very radical “abolitionist” type platform had been written by the Republican Party in 1854. Douglas accused Lincoln of taking the side of the common enemy in the Mexican War. ...


In [5]:
# Extrait fidele de 05_RAG_Modern.ipynb (cellule 17) - strategies de chunking
def chunk_fixed(text: str, chunk_size: int = 400, overlap: int = 50) -> List[str]:
    """
    Chunking fixe par nombre de mots.
    
    Args:
        text: Texte à découper
        chunk_size: Nombre de mots par chunk
        overlap: Chevauchement entre chunks (évite les coupures brutales)
    """
    words = text.split()
    chunks = []
    start = 0
    
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += (chunk_size - overlap)
    
    return chunks


def chunk_semantic(text: str, max_sentences: int = 5) -> List[str]:
    """
    Chunking sémantique par phrases.
    Regroupe les phrases en chunks cohérents.
    """
    import re
    # Découpage par phrases (approximatif)
    sentences = re.split(r'(?<=[.!?])\s+', text)
    
    chunks = []
    current_chunk = []
    
    for sentence in sentences:
        current_chunk.append(sentence)
        if len(current_chunk) >= max_sentences:
            chunks.append(" ".join(current_chunk))
            current_chunk = []
    
    # Dernier chunk
    if current_chunk:
        chunks.append(" ".join(current_chunk))
    
    return chunks


def chunk_recursive(text: str, max_size: int = 500, delimiters: List[str] = None) -> List[str]:
    """
    Chunking récursif avec délimiteurs hiérarchiques.
    Essaie de découper par paragraphes, puis phrases, puis mots.
    """
    if delimiters is None:
        delimiters = ["\n\n", "\n", ". ", " "]
    
    if len(text) <= max_size or not delimiters:
        return [text] if text.strip() else []
    
    delimiter = delimiters[0]
    parts = text.split(delimiter)
    
    chunks = []
    current = ""
    
    for part in parts:
        if len(current) + len(part) + len(delimiter) <= max_size:
            current += (delimiter if current else "") + part
        else:
            if current:
                chunks.append(current)
            # Récursion avec délimiteur suivant si le part est trop grand
            if len(part) > max_size:
                chunks.extend(chunk_recursive(part, max_size, delimiters[1:]))
            else:
                current = part
    
    if current:
        chunks.append(current)
    
    return chunks


# Comparaison des stratégies
print("=== Comparaison des stratégies de chunking ===")
print(f"\nTexte source: {len(debate_text)} caractères")

chunks_fixed = chunk_fixed(debate_text, chunk_size=100, overlap=20)
chunks_semantic = chunk_semantic(debate_text, max_sentences=3)
chunks_recursive = chunk_recursive(debate_text, max_size=500)

print(f"\nChunking fixe (100 mots, overlap 20): {len(chunks_fixed)} chunks")
print(f"Chunking sémantique (3 phrases): {len(chunks_semantic)} chunks")
print(f"Chunking récursif (max 500 chars): {len(chunks_recursive)} chunks")

# Création du DataFrame de chunks pour la suite
chunks = chunk_fixed(debate_text, chunk_size=400, overlap=50)

df_chunks = pd.DataFrame({
    "chunk_id": range(len(chunks)),
    "source": "Lincoln-Douglas Debate 1 (Ottawa, 1858)",
    "text": chunks
})

print(f"Base de connaissances: {len(df_chunks)} chunks")
df_chunks.head()

=== Comparaison des stratégies de chunking ===

Texte source: 98529 caractères

Chunking fixe (100 mots, overlap 20): 222 chunks
Chunking sémantique (3 phrases): 214 chunks
Chunking récursif (max 500 chars): 327 chunks
Base de connaissances: 51 chunks


,chunk_id,source,text
0,0,"Lincoln-Douglas Debate 1 (Ottawa, 1858)","First Debate: Ottawa, Illinois August 21, 1858..."
1,1,"Lincoln-Douglas Debate 1 (Ottawa, 1858)",were proclaimed wherever the Constitution rule...
2,2,"Lincoln-Douglas Debate 1 (Ottawa, 1858)",the Whig party and the Democratic party both s...
3,3,"Lincoln-Douglas Debate 1 (Ottawa, 1858)",name and disguise of a Republican party. (Laug...
4,4,"Lincoln-Douglas Debate 1 (Ottawa, 1858)",with such views as the circumstances and exige...


In [6]:
# Extrait fidele de 05_RAG_Modern.ipynb (cellule 29) - embeddings
def create_embedding(text: str, model: str = "text-embedding-3-large") -> List[float]:
    """
    Génère un embedding pour un texte donné.
    
    Args:
        text: Texte à vectoriser
        model: Modèle d'embedding OpenAI
    
    Returns:
        Vecteur d'embedding (liste de floats)
    """
    try:
        response = client.embeddings.create(
            model=model,
            input=[text]
        )
        return response.data[0].embedding
    except Exception as e:
        print(f"Erreur embedding: {e}")
        return None


def create_embeddings_batch(texts: List[str], model: str = "text-embedding-3-large") -> List[List[float]]:
    """
    Génère des embeddings en batch (plus efficace).
    
    Note: OpenAI supporte jusqu'à 2048 textes par requête.
    """
    try:
        response = client.embeddings.create(
            model=model,
            input=texts
        )
        return [item.embedding for item in response.data]
    except Exception as e:
        print(f"Erreur batch embedding: {e}")
        return [None] * len(texts)


# Génération des embeddings pour tous les chunks
print("Génération des embeddings...")
embeddings = create_embeddings_batch(df_chunks["text"].tolist())
df_chunks["embedding"] = embeddings

print(f"Embeddings générés: {len([e for e in embeddings if e])} / {len(embeddings)}")
print(f"Dimension des vecteurs: {len(embeddings[0]) if embeddings[0] else 'N/A'}")

Génération des embeddings...


Embeddings générés: 51 / 51
Dimension des vecteurs: 3072


In [7]:
# Extrait fidele de 05_RAG_Modern.ipynb (cellule 36) - VectorStore : index k-NN
from sklearn.neighbors import NearestNeighbors

class VectorStore:
    """
    Base vectorielle simple basée sur scikit-learn.
    En production, utiliser Pinecone, Qdrant, Weaviate, ou Chroma.
    """
    
    def __init__(self, df: pd.DataFrame, embedding_col: str = "embedding"):
        self.df = df
        self.embedding_col = embedding_col
        
        # Filtrer les embeddings valides
        valid_mask = df[embedding_col].apply(lambda x: x is not None)
        self.valid_indices = df[valid_mask].index.tolist()
        
        # Construire la matrice de vecteurs
        vectors = np.array(df.loc[self.valid_indices, embedding_col].tolist())
        
        # Index k-NN
        self.nn = NearestNeighbors(n_neighbors=min(5, len(vectors)), metric="cosine")
        self.nn.fit(vectors)
        
        print(f"VectorStore initialisé: {len(self.valid_indices)} vecteurs")
    
    def search(self, query_embedding: List[float], k: int = 3) -> List[Dict[str, Any]]:
        """
        Recherche les k chunks les plus similaires.
        
        Returns:
            Liste de dicts avec chunk_id, text, source, score
        """
        distances, indices = self.nn.kneighbors([query_embedding], n_neighbors=k)
        
        results = []
        for i, (dist, idx) in enumerate(zip(distances[0], indices[0])):
            original_idx = self.valid_indices[idx]
            row = self.df.iloc[original_idx]
            results.append({
                "chunk_id": row["chunk_id"],
                "text": row["text"],
                "source": row["source"],
                "score": 1 - dist  # Cosine similarity (1 = identique)
            })
        
        return results


# Initialisation du VectorStore
vector_store = VectorStore(df_chunks)

VectorStore initialisé: 51 vecteurs


In [8]:
# Extrait fidele de 05_RAG_Modern.ipynb (cellule 44) - generation RAG avec sources citees
def rag_query(question: str, vector_store: VectorStore, k: int = 3) -> Dict[str, Any]:
    """
    Pipeline RAG complet avec Chat Completions.
    
    Args:
        question: Question de l'utilisateur
        vector_store: Base vectorielle
        k: Nombre de chunks à récupérer
    
    Returns:
        Dict avec réponse, sources, et métadonnées
    """
    # 1. Retrieval
    query_embedding = create_embedding(question)
    chunks = vector_store.search(query_embedding, k=k)
    
    # 2. Construction du contexte
    context = "\n\n".join([
        f"[Source: {c['source']} | Chunk {c['chunk_id']}]\n{c['text']}"
        for c in chunks
    ])
    
    # 3. Prompt augmenté
    system_prompt = """Tu es un assistant expert en histoire américaine.
Réponds aux questions en te basant UNIQUEMENT sur le contexte fourni.
Si l'information n'est pas dans le contexte, dis-le clairement.
Cite tes sources avec le format [Chunk X]."""
    
    user_prompt = f"""Contexte:
{context}

Question: {question}

Réponds de manière précise en citant les sources."""
    
    # 4. Génération
    # Note: temperature not supported by gpt-5-mini, uses default (1.0)
    # Budget relevé à 2000 : gpt-5-mini est un modèle de raisonnement dont les
    # tokens de raisonnement sont déduits de max_completion_tokens. Avec 500,
    # tout le budget pouvait être consommé par le raisonnement sans texte final
    # visible (issue #3571). 2000 laisse une marge confortable pour le contenu.
    response = client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        max_completion_tokens=2000
    )
    
    return {
        "question": question,
        "answer": response.choices[0].message.content,
        "sources": chunks,
        "model": DEFAULT_MODEL,
        "tokens": {
            "prompt": response.usage.prompt_tokens,
            "completion": response.usage.completion_tokens
        }
    }


# Test du pipeline RAG
result = rag_query(
    "What were Lincoln's main arguments about slavery in the debate?",
    vector_store
)

print("=== Réponse RAG ===")
print(f"\nQuestion: {result['question']}")
print(f"\nRéponse:\n{result['answer']}")
print(f"\nTokens utilisés: {result['tokens']}")

=== Réponse RAG ===

Question: What were Lincoln's main arguments about slavery in the debate?

Réponse:
Voici, d’après le contexte fourni, les principaux arguments de Lincoln sur l’esclavage pendant le débat :

- « La maison divisée » : Lincoln affirmait que le gouvernement « ne peut pas durer éternellement moitié esclave, moitié libre » ; il prédit que la nation deviendrait « tout l’une chose, ou tout l’autre » — soit les opposants à l’esclavage arrêteraient son expansion et le placeraient sur la voie de l’extinction, soit ses partisans le pousseraient jusqu’à le rendre légal dans tous les États. [Chunk 9]

- Accusation contre Douglas : Lincoln reprocha à Douglas de chercher à « nationaliser » l’esclavage (c’est‑à‑dire d’étendre et d’assurer la légalité de l’esclavage à l’échelle nationale). [Chunk 0]

- Défense et réfutation : Lincoln passa une bonne partie de son temps à nier les accusations de Douglas (que Lincoln serait un abolitionniste radical, qu’il avait « abolitionisé » les 

### Lecture de l'extraction

Rien n'a change dans le calcul : les fonctions sont celles du 05, au caractere pres. Ce qui a change est la **forme** : regroupees ici, elles forment le module implicite que le 05 portait epars. C'est la premiere etape du passage a l'application - et elle est deja faite dans nos series chaque fois qu'une cellule ne fait qu'appeler des fonctions definies plus tot.

## Servir : le pipeline devient une API

Une API transforme une fonction en service : adresse, contrat d'entree/sortie, disponibilite. Le point de design qui tient tout : **l'application ne possede aucune logique RAG** - elle appelle `rag_query` exactement comme le ferait une cellule du 05. Une seule source de verite, deux canaux (notebook, HTTP).

In [9]:
# L'application : zero logique RAG, seulement le contrat HTTP
from pydantic import BaseModel

def _native(value):
    """Normalise les scalaires numpy du pipeline en natifs JSON (pydantic v2 les refuse)."""
    if isinstance(value, dict):
        return {k: _native(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_native(v) for v in value]
    if hasattr(value, "item"):  # numpy.int64, numpy.float64, ...
        return value.item()
    return value

class RagAnswer(BaseModel):
    question: str
    answer: str
    sources: list
    latency_ms: float

app = FastAPI(title="RAG Lincoln-Douglas", version="0.1.0")

@app.get("/health")
def health():
    return {"status": "ok", "chunks": len(vector_store.df) if hasattr(vector_store, "df") else None}

@app.get("/query", response_model=RagAnswer)
def query(q: str, k: int = 3):
    t0 = time.perf_counter()
    result = rag_query(q, vector_store, k=k)
    return RagAnswer(
        question=q,
        answer=result["answer"] if isinstance(result, dict) else str(result),
        sources=_native(result.get("sources", [])) if isinstance(result, dict) else [],
        latency_ms=round((time.perf_counter() - t0) * 1000, 1),
    )

print("Application definie :", [r.path for r in app.routes if hasattr(r, "path")])

Application definie : ['/openapi.json', '/docs', '/docs/oauth2-redirect', '/redoc', '/health', '/query']


In [10]:
# Service local sur un port libre : aucun port en dur, aucun processus orphelin
def free_port():
    s = socket.socket()
    s.bind(("127.0.0.1", 0))
    port = s.getsockname()[1]
    s.close()
    return port

API_PORT = free_port()
config = uvicorn.Config(app, host="127.0.0.1", port=API_PORT, log_level="warning")
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()

base = f"http://127.0.0.1:{API_PORT}"
health = None
with httpx.Client(base_url=base, timeout=60.0) as probe:
    for _ in range(50):
        try:
            r = probe.get("/health")
            if r.status_code == 200:
                health = r.json()
                break
        except httpx.ConnectError:
            time.sleep(0.2)
print("API en service sur", base, "-", health)

API en service sur http://127.0.0.1:44148 - {'status': 'ok', 'chunks': 51}


In [11]:
# Appels reels : le contrat tient-il sur de vraies questions ?
QUESTIONS = [
    "What did Lincoln argue about slavery in the territories?",
    "What position did Douglas defend on popular sovereignty?",
    "What did the two candidates disagree on about the Dred Scott decision?",
]
api_latencies = []
with httpx.Client(base_url=base, timeout=60.0) as http:
    for q in QUESTIONS:
        t0 = time.perf_counter()
        r = http.get("/query", params={"q": q})
        r.raise_for_status()
        api_latencies.append((time.perf_counter() - t0) * 1000)
        body = r.json()
        print(f"[{body['latency_ms']:7.1f} ms cote serveur] {q[:52]}...")
        print(f"           -> {body['answer'][:90]}...")

[ 8693.8 ms cote serveur] What did Lincoln argue about slavery in the territor...
           -> Lincoln soutenait principalement deux idées au sujet de l’esclavage dans les territoires, ...


[ 6300.2 ms cote serveur] What position did Douglas defend on popular sovereig...
           -> Selon le contexte fourni, Douglas défendait la « popular sovereignty » — c’est‑à‑dire que ...


[ 9755.1 ms cote serveur] What did the two candidates disagree on about the Dr...
           -> Ils étaient en désaccord sur l’acceptation et les effets de la décision Dred Scott. D’aprè...


### Lecture des premieres mesures

Le temps de reponse cote serveur (`latency_ms`, mesure dans le handler) et le temps cote client (autour de l'appel HTTP) ne sont pas le meme nombre : la difference est le **cout du transport** - serialization, boucle HTTP, file du thread Uvicorn. Le service ne rend pas le pipeline plus rapide ; il le rend **partageable**.

## Interfacer : une UI qui parle a l'API

L'interface ne connait ni le corpus, ni le VectorStore, ni la cle d'API : elle ne connait que l'URL du service. C'est la regle qui garde une seule source de verite - et le test qu'elle tient vraiment : couper l'API, et l'UI doit echouer proprement.

In [12]:
# UI Gradio sur l'API - jamais sur le pipeline
UI_PORT = free_port()

def ask_via_api(question: str):
    # Passe par l'API - jamais par le pipeline. Rend (reponse, sources brutes).
    with httpx.Client(base_url=base, timeout=60.0) as http:
        r = http.get("/query", params={"q": question})
        r.raise_for_status()
        body = r.json()
    return body["answer"], body["sources"]

ui = gr.Interface(
    fn=ask_via_api,
    inputs=gr.Textbox(label="Question sur le debat Lincoln-Douglas",
                      value=QUESTIONS[0]),
    outputs=[gr.Textbox(label="Reponse"), gr.JSON(label="Sources")],
    title="RAG Lincoln-Douglas (sert par FastAPI)",
    flagging_mode="never",
)
ui.launch(prevent_thread_lock=True, quiet=True, server_port=UI_PORT)
print("UI en service sur http://127.0.0.1:%d - l'UI passe par l'API, pas par le pipeline" % UI_PORT)

UI en service sur http://127.0.0.1:45100 - l'UI passe par l'API, pas par le pipeline


In [13]:
# Le test du canal unique : memes sources par les deux canaux ?
# La generation LLM varie d'un appel a l'autre ; ce qui est deterministe, c'est
# la recherche : meme question -> meme embedding -> memes chunks. Le test la porte.
_, sources_ui = ask_via_api(QUESTIONS[0])
with httpx.Client(base_url=base, timeout=60.0) as http:
    sources_api = http.get("/query", params={"q": QUESTIONS[0]}).json()["sources"]
print("UI et API rendent les memes sources :", list(sources_ui) == list(sources_api))

UI et API rendent les memes sources : True


## Mesurer : ce que coute chaque couche

Meme lot de questions, deux chemins - la fonction directe dans le notebook, l'appel HTTP vers l'API - plus une reference : le transport pur (GET /health, sans generation). Chaque chemin paie un **tour de chauffe** avant la mesure : le premier appel absorbe la connexion et le chargement du modele, qui ne se repetent plus ensuite. L'embedding de la question, lui, est recalcule a chaque appel de `rag_query` — le carnet n'installe aucun cache local — et il est donc paye a chaque tour, identiquement sur les deux chemins.

In [14]:
import pandas as pd

# Transport pur : /health ne genere rien - il isole le cout du va-et-vient HTTP
with httpx.Client(base_url=base, timeout=10.0) as http:
    health_lat = []
    for _ in range(20):
        t0 = time.perf_counter()
        r = http.get("/health")
        r.raise_for_status()
        health_lat.append((time.perf_counter() - t0) * 1000)
print(f"Transport pur (GET /health, 20 appels) : mediane {statistics.median(health_lat):.1f} ms")

def bench(question, k=3, runs=3):
    """Latences medians sur `runs` repetitions, apres un tour de chauffe par chemin."""
    rag_query(question, vector_store, k=k)  # chauffe : hors du chrono, absorbe connexion et chargement du modele
    direct = []
    for _ in range(runs):
        t0 = time.perf_counter()
        rag_query(question, vector_store, k=k)
        direct.append((time.perf_counter() - t0) * 1000)
    with httpx.Client(base_url=base, timeout=60.0) as http:
        http.get("/query", params={"q": question, "k": k})  # chauffe
        via_api = []
        for _ in range(runs):
            t0 = time.perf_counter()
            r = http.get("/query", params={"q": question, "k": k})
            r.raise_for_status()
            via_api.append((time.perf_counter() - t0) * 1000)
    return {
        "question": question[:38] + "...",
        "direct p50 (ms)": round(statistics.median(direct), 1),
        "api p50 (ms)": round(statistics.median(via_api), 1),
        "surcout api (ms)": round(statistics.median(via_api) - statistics.median(direct), 1),
    }

rows = [bench(q, runs=3) for q in QUESTIONS]
df_bench = pd.DataFrame(rows)
print(df_bench.to_string(index=False))

Transport pur (GET /health, 20 appels) : mediane 2.0 ms


                                 question  direct p50 (ms)  api p50 (ms)  surcout api (ms)
What did Lincoln argue about slavery i...           9362.2        8576.3            -785.9
What position did Douglas defend on po...           5415.3        5905.7             490.4
What did the two candidates disagree o...          10136.9        9596.1            -540.8


### Lecture du cout du service

Le resultat le plus instructif est negatif : le surcout de l'API peut etre **negatif** - l'appel via HTTP parfois plus rapide que l'appel direct. Rien de magique : la generation LLM dure des secondes et varie d'un appel a l'autre, et cette variance ecrase le transport. La ligne /health donne la verite du transport seul : de l'ordre de la milliseconde, sans generation. La conclusion pratique pour la serie : le passage au service est une decision d'usage (qui peut appeler), pas une decision de performance - a l'echelle ou l'on mesure, servir ne coute rien de discernable.

## Exercices

Trois exercices pour s'approprier chaque couche. Les stubs s'executent sans erreur (regle C.1) : ils affichent ce qu'il reste a faire.

In [15]:
# Exercice 1 - Un endpoint de supervision /chunks
# TODO etudiant : ajouter un endpoint GET /chunks qui rend le nombre de chunks,
# leur taille moyenne et les 3 premiers caracteres de chacun (max 5).
# Indice : decorer une fonction comme /health, lire vector_store.
# Etape 1 : definir la fonction ci-dessous puis l'enregistrer sur app
def chunks_summary():
    # TODO etudiant
    print("Exercice a completer : endpoint /chunks")
    return None

result_exo_1 = None  # TODO etudiant : appeler l'endpoint une fois enregistre

In [16]:
# Exercice 2 - Un curseur k dans l'UI
# TODO etudiant : ajouter un Slider k (1 a 6, defaut 3) entre la question et l'API,
# et faire passer k au parametre de /query.
# Indice : gr.Slider(...), puis un deuxieme argument dans ask_via_api.
# Etape 1 : completer la fonction ci-dessous
def ask_with_k(question, k=3):
    # TODO etudiant
    print("Exercice a completer : ask_with_k")
    return None

result_exo_2 = None  # TODO etudiant

In [17]:
# Exercice 3 - Une charge legere : 10 requetes, verdict
# TODO etudiant : envoyer 10 fois la meme question a /query, mesurer chaque latence
# cote client, rendre min / mediane / max en millisecondes et le taux d'erreur HTTP.
# Indice : httpx.Client, time.perf_counter, statistics.median.
# Etape 1 : completer la fonction ci-dessous
def light_load(question, n=10):
    # TODO etudiant
    print("Exercice a completer : light_load")
    return None

result_exo_3 = None  # TODO etudiant

## Conclusion : ce que le notebook gagne a devenir application

- **Extraction** : le pipeline du 05 devient un module implicite - les fonctions existaient deja, elles sont simplement honnetement regroupees ;
- **Service** : une API FastAPI locale ajoute un contrat d'appel sans toucher a la logique - le transport pur se mesure en millisecondes, noye dans la variance de la generation ;
- **Interface** : l'UI Gradio ne connait que l'URL de l'API - une seule source de verite, deux canaux.

La limite honnete de l'exercice : ce service vit le temps du notebook (arret propre ci-dessous). Le passage reel a la production ajouterait la persistence, l'authentification et le deploiement - hors perimetre de cette serie, mais le chemin jusqu'ici est desormais balise.

In [18]:
# Arret propre des services : aucun processus orphelin apres le carnet
try:
    ui.close()
    print("UI Gradio : arretee")
except Exception as e:
    print("UI deja arretee :", e)
server.should_exit = True
thread.join(timeout=5)
print("API FastAPI : arretee" if not thread.is_alive() else "API : thread encore vivant")
print("=== Recapitulatif ===")
print("pipeline extrait de 05 | API servie sur port", API_PORT, "| UI sur port", UI_PORT,
      "| surcout api median :",
      (df_bench["surcout api (ms)"].median() if "df_bench" in dir() else "n/a"), "ms")

Closing server running on port: 45100
UI Gradio : arretee
API FastAPI : arretee
=== Recapitulatif ===
pipeline extrait de 05 | API servie sur port 44148 | UI sur port 45100 | surcout api median : -540.8 ms
